# Condition Testing Colab

Notebook pentru generare predicții + evaluare pe `TEST_COMPLETE_PAIR_IDS`.

Acoperă acum:
- eval rapid pentru condiții care au deja predicții în repo (`cond1`, `cond2`);
- condiția 5: Qwen3-4B + LoRA adapter fine-tuned pe synthetic Claude.

Nu folosește pool/ICL examples ca material de test. Test IDs vin din `src/data_split.py`.

In [ ]:
# Dependencies for local-model inference + eval.
%pip -q install -U \
  "transformers>=4.57.0,<5" \
  "accelerate>=1.11.0" \
  "peft>=0.17.0" \
  "bitsandbytes>=0.45.0" \
  "jsonschema" \
  "sentencepiece" \
  "protobuf" \
  "bert-score"

In [ ]:
# Repo checkout / refresh.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SoftNestSol/Medical-Notes.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Medical-Notes")
UPDATE_REPO = True

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    elif UPDATE_REPO:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    os.chdir(REPO_DIR)
else:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "src" / "data_split.py").exists():
            os.chdir(candidate)
            break

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
SOTA = SRC / "SOTA_EVALUATION"
ICL = SRC / "ICL"
SCRIPTS = ROOT / "scripts"
for path in [SRC, SOTA, ICL, SCRIPTS]:
    sys.path.insert(0, str(path))

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
print("ROOT", ROOT)

In [ ]:
# Config.
from data_split import TEST_COMPLETE_PAIR_IDS

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
CONDITION_NAME = "cond5_qwen3_4b_synth_claude_ft"
PRED_DIR = ROOT / "data" / "chiropractor_ro" / "predictions" / CONDITION_NAME
REF_DIR = ROOT / "data" / "chiropractor_ro" / "refs"
CONV_DIR = ROOT / "data" / "chiropractor_ro" / "conversations"

# Empty RUN_NAME = auto-pick latest Drive run containing an adapter folder.
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/Medical-Notes/ft-runs")
RUN_NAME = ""
LOCAL_ADAPTER_DIR = Path("/content/ft_adapter")

TEST_IDS = sorted(TEST_COMPLETE_PAIR_IDS, key=lambda x: int(x.replace("audio", "")))
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 1024
OVERWRITE_PREDICTIONS = False

# BERTScore downloads a Romanian encoder and is slower. Use False for quick structural checks.
RUN_BERTSCORE = False

print("TEST_IDS", len(TEST_IDS), TEST_IDS)
print("PRED_DIR", PRED_DIR)

In [ ]:
# Optional: evaluate existing prediction directories, e.g. cond1/cond2.
EXISTING_CONDITIONS = [
    "cond1_gemini_zeroshot",
    "cond2_claude_zeroshot",
]

for condition in EXISTING_CONDITIONS:
    pred_dir = ROOT / "data" / "chiropractor_ro" / "predictions" / condition
    if not pred_dir.exists():
        print(condition, "missing, skip")
        continue
    cmd = [
        sys.executable,
        str(SOTA / "eval.py"),
        "--pred-dir", str(pred_dir),
        "--ref-dir", str(REF_DIR),
        "--split", "test-complete",
    ]
    if not RUN_BERTSCORE:
        cmd.append("--skip-bertscore")
    print("RUN", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# Locate/copy LoRA adapter from Drive.
import shutil

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped/failed:", repr(exc))

def find_adapter_dir() -> Path:
    if RUN_NAME:
        candidate = DRIVE_RUN_ROOT / RUN_NAME / "adapter"
        if not candidate.exists():
            raise FileNotFoundError(candidate)
        return candidate
    candidates = sorted(
        [path for path in DRIVE_RUN_ROOT.glob("*/adapter") if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No adapter folders under {DRIVE_RUN_ROOT}")
    return candidates[0]

adapter_source = find_adapter_dir()
if LOCAL_ADAPTER_DIR.exists():
    shutil.rmtree(LOCAL_ADAPTER_DIR)
shutil.copytree(adapter_source, LOCAL_ADAPTER_DIR)
print("adapter source", adapter_source)
print("local adapter", LOCAL_ADAPTER_DIR)
print("files", sorted(path.name for path in LOCAL_ADAPTER_DIR.iterdir()))

In [ ]:
# Load base model + LoRA adapter for condition 5 inference.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    token=False,
)
model = PeftModel.from_pretrained(base_model, str(LOCAL_ADAPTER_DIR))
model.eval()
print("loaded model + adapter")

In [ ]:
# Prediction helpers.
import json
import re
from build_chiro_sft_jsonl import SYSTEM_PROMPT
from build_real_icl_examples import read_conversation_as_transcript
from parser import ParseError, SchemaError, parse_note

CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}

def make_messages(transcript: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "--- TRANSCRIEREA CONSULTAȚIEI ---\n\n" + transcript.strip()},
    ]

def generate_raw(transcript: str) -> str:
    prompt = tokenizer.apply_chat_template(
        make_messages(transcript),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def write_note(path: Path, note: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(note, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

In [ ]:
# Generate condition 5 predictions on TEST_COMPLETE_PAIR_IDS.
failures = []
PRED_DIR.mkdir(parents=True, exist_ok=True)

for idx, conv_id in enumerate(TEST_IDS, start=1):
    out_path = PRED_DIR / f"{conv_id}.json"
    raw_path = PRED_DIR / f"{conv_id}.raw.txt"
    if out_path.exists() and not OVERWRITE_PREDICTIONS:
        print(f"[{idx}/{len(TEST_IDS)}] {conv_id} skip")
        continue

    transcript = read_conversation_as_transcript(CONV_DIR / f"{conv_id}.json")
    raw = generate_raw(transcript)
    raw_path.write_text(raw, encoding="utf-8")
    try:
        note = parse_note(raw)
    except (ParseError, SchemaError) as exc:
        print(f"[{idx}/{len(TEST_IDS)}] {conv_id} FAIL {type(exc).__name__}: {exc}")
        failures.append({"id": conv_id, "error": f"{type(exc).__name__}: {exc}", "raw_path": str(raw_path)})
        continue
    write_note(out_path, note)
    print(f"[{idx}/{len(TEST_IDS)}] {conv_id} saved")

(PRED_DIR / "_failures.json").write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
print("done", len(TEST_IDS) - len(failures), "ok", len(failures), "failed")
print("PRED_DIR", PRED_DIR)

In [ ]:
# Evaluate condition 5 predictions.
cmd = [
    sys.executable,
    str(SOTA / "eval.py"),
    "--pred-dir", str(PRED_DIR),
    "--ref-dir", str(REF_DIR),
    "--split", "test-complete",
]
if not RUN_BERTSCORE:
    cmd.append("--skip-bertscore")
print("RUN", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Compact comparison table from available eval summaries.
import pandas as pd

def get_nested(d, keys):
    cur = d
    for key in keys:
        if cur is None or key not in cur:
            return None
        cur = cur[key]
    return cur

rows = []
for pred_dir in sorted((ROOT / "data" / "chiropractor_ro" / "predictions").glob("cond*")):
    summary_path = pred_dir / "eval" / "summary.json"
    if not summary_path.exists():
        continue
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    rows.append({
        "condition": pred_dir.name,
        "n_cases": summary.get("n_cases"),
        "vas_acc": get_nested(summary, ["fields", "evaluarea_durerii_vas", "accuracy"]),
        "loc_micro_f1": get_nested(summary, ["fields", "localizarea_durerii", "micro_f1"]),
        "loc_exact": get_nested(summary, ["fields", "localizarea_durerii", "exact_match_accuracy"]),
        "ant_micro_f1": get_nested(summary, ["fields", "antecedente", "micro_f1"]),
        "med_name_f1": get_nested(summary, ["fields", "medicatie_actuala", "name_micro_f1"]),
        "eval_func_bert": get_nested(summary, ["fields", "evaluare_functionala_initiala", "bertscore_f1_mean"]),
    })

df = pd.DataFrame(rows).sort_values("condition")
display(df)
out = ROOT / "artifacts" / "eval" / "condition_comparison.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print("wrote", out)